## Basline Models Checks

**import**

In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Load data**

In [74]:
DATASET="../dataset/SmartHomeIoTNLU.csv"

df=pd.read_csv(DATASET)


**Dataset Split**

In [75]:
from sklearn.model_selection import train_test_split

events=df.Event_ID.unique()

train_events,test_events=train_test_split(events,test_size=0.15,random_state=42)

train=df[df.Event_ID.isin(train_events)]

test=df[df.Event_ID.isin(test_events)]


## Intent Classification, TF-IDF + Logistic Regression

In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer=TfidfVectorizer()

X_train=vectorizer.fit_transform(train.Command)

X_test=vectorizer.transform(test.Command)

model=LogisticRegression()

model.fit(X_train,train.Intent_Type)

pred=model.predict(X_test)

In [77]:
# Metrics:

from sklearn.metrics import classification_report

print(classification_report(test.Intent_Type,pred))

                       precision    recall  f1-score   support

Direct_Device_Control       0.99      0.99      0.99     19143
        Scene_Control       1.00      1.00      1.00     92169

             accuracy                           1.00    111312
            macro avg       0.99      1.00      1.00    111312
         weighted avg       1.00      1.00      1.00    111312



## Device Multi-label Prediction

In [78]:
device_events = (
    df.groupby("Event_ID")["Device"]
    .apply(list)
)

device_events.head()

Event_ID
000007dd-0fbe-4499-94b3-3cbf4fe3b7d4                                                 [AC]
000014e4-e98a-4eb1-8bc1-3387b71c7108                                              [Light]
00004fc4-d95e-4495-b445-8c8678fd874a    [SmartPlug_Air Purifier, SmartPlug_Desk Lamp, ...
000073db-2ff3-4058-843e-8093a071a516                                              [Light]
000131bf-4b5a-4335-9626-31c9f02ae7b8    [Exhaust / Vent, SmartPlug_Coffee Maker, Smart...
Name: Device, dtype: object

In [79]:
# Convert devices into multi-label vectors

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

Y = mlb.fit_transform(
    device_events
)

print(Y.shape)

(200320, 19)


In [80]:
# Split into train/test

from sklearn.model_selection import train_test_split


X = device_events.index


X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.15,
    random_state=42
)


In [81]:
# Create command features

commands = (
    df.groupby("Event_ID")["Command"]
    .first()
)


from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer()


X_text = vectorizer.fit_transform(
    commands
)



In [82]:
# Split text features

X_train_text, X_test_text, Y_train, Y_test = train_test_split(
    X_text,
    Y,
    test_size=0.15,
    random_state=42
)

In [83]:
# Train multi-label classifier

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier


model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000
    )
)

model.fit(
    X_train_text,
    Y_train
)

,estimator,LogisticRegre...max_iter=1000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [84]:
# Generate predictions

prediction = model.predict(
    X_test_text
)

In [85]:
#Metrics

from sklearn.metrics import f1_score


f1 = f1_score(
    Y_test,
    prediction,
    average="samples"
)


print(
    "Sample F1:",
    f1
)

Sample F1: 0.9705701927333098


In [86]:
# Add all device prediction metrics

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss
)


results = {

"Precision":
precision_score(
    Y_test,
    prediction,
    average="samples"
),

"Recall":
recall_score(
    Y_test,
    prediction,
    average="samples"
),

"F1-score":
f1_score(
    Y_test,
    prediction,
    average="samples"
),

"Hamming Loss":
hamming_loss(
    Y_test,
    prediction
)

}


results

/home3/ykwx38/myjupyterenv4/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'Precision': 0.9762333105858465,
 'Recall': 0.9701304003245896,
 'F1-score': 0.9705701927333098,
 'Hamming Loss': 0.010919371111484782}

## Temperature Parameter Prediction

In [87]:
# Filter AC records

import pandas as pd

# Load dataset
df = pd.read_csv("../dataset/SmartHomeIoTNLU.csv")

# Keep only AC rows
ac_df = df[df["Device"] == "AC"].copy()

print(f"Number of AC samples: {len(ac_df)}")

ac_df.head()

Number of AC samples: 105520


,Event_ID,Intent_Type,Command,Device,Current State,Time,Light,Temperature,Noise,Occupancy,Location,Scenario,Action,Parameters
11,39e4d0b0-7f65-4089-a6e7-3c2a7ddf7677,Scene_Control,Relax before bed,AC,Off,Afternoon,77%,30°C,Low,2,Bedroom,Sleep Preparation_Bedroom,Adjust,Temp=25°C
14,b556c2f3-4287-4e45-8c06-9676f199b305,Direct_Device_Control,Can you shut the cooling down,AC,Off,Afternoon,84%,20°C,Medium,2,Living Room,Direct_Control,Off,Temp=20°C
17,dd653c19-f6ad-4378-8105-71243d05bc77,Direct_Device_Control,Climate control on,AC,Adjust,Night,11%,18°C,Low,1,Bedroom,Direct_Control,Adjust,Temp=23°C
27,b869b4ab-a63a-4f41-a441-78640d15ac4c,Scene_Control,Reset my study space,AC,Adjust,Night,39%,20°C,Low,1,Study Room,Cleaning_Study Room,Adjust,Temp=24°C
38,ac79150e-86bd-4586-aa15-38bdb55e749c,Direct_Device_Control,Lower the room heat,AC,Off,Morning,80%,16°C,Low,2,Kitchen,Direct_Control,Off,Temp=16°C


In [88]:
# Convert the parameter to a numeric temperature

ac_df["Target_Temperature"] = (
    ac_df["Parameters"]
    .str.extract(r'(\d+)')
    .astype(float)
)

ac_df[["Parameters", "Target_Temperature"]].head()

,Parameters,Target_Temperature
11,Temp=25°C,25.0
14,Temp=20°C,20.0
17,Temp=23°C,23.0
27,Temp=24°C,24.0
38,Temp=16°C,16.0


In [89]:
# Create input features

from sklearn.feature_extraction.text import TfidfVectorizer

# Combine command and context
ac_df["Input_Text"] = (
    ac_df["Command"] + " " +
    ac_df["Time"] + " " +
    ac_df["Location"] + " " +
    ac_df["Noise"]
)

vectorizer = TfidfVectorizer(max_features=1000)

X_text = vectorizer.fit_transform(ac_df["Input_Text"])

In [90]:
# Add numerical features

import numpy as np

# Light: "43%" -> 43
ac_df["Light"] = (
    ac_df["Light"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

# Temperature: "16°C" -> 16
ac_df["Temperature"] = (
    ac_df["Temperature"]
    .str.extract(r'(\d+)')
    .astype(float)
)

In [91]:
# Encode categorical variables

from sklearn.preprocessing import LabelEncoder

time_encoder = LabelEncoder()
room_encoder = LabelEncoder()
noise_encoder = LabelEncoder()

ac_df["Time_ID"] = time_encoder.fit_transform(ac_df["Time"])
ac_df["Room_ID"] = room_encoder.fit_transform(ac_df["Location"])
ac_df["Noise_ID"] = noise_encoder.fit_transform(ac_df["Noise"])

In [92]:
# Combine all features

from scipy.sparse import hstack

numeric = ac_df[
    [
        "Temperature",
        "Light",
        "Occupancy",
        "Time_ID",
        "Room_ID",
        "Noise_ID"
    ]
].values

X = hstack([X_text, numeric])

y = ac_df["Target_Temperature"]

In [93]:
# Train/Test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42
)

In [94]:
# Train a baseline regressor
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [95]:
# Predict
prediction = model.predict(X_test)

In [96]:
# Evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, prediction)

rmse = np.sqrt(
    mean_squared_error(y_test, prediction)
)

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")

MAE  : 0.669
RMSE : 1.088


In [97]:
# Display prediction examples

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": prediction
})

results.head(20)

,Actual,Predicted
0,23.0,22.080000
1,21.0,22.880000
2,24.0,23.620000
3,18.0,18.000000
4,27.0,26.815000
5,15.0,15.000000
6,25.0,22.600000
7,24.0,21.842500
8,23.0,23.111667
9,15.0,15.000000
